In [1]:
import scanpy as sc
import anndata



import importlib

import pandas as pd
import numpy as np
#import scanpy as sc
#import scycle as cc
# import scvelo as sv
#import anndata
from sklearn.decomposition import PCA

import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

In [2]:
adata = sc.read_h5ad("data/SKM_human.h5ad")
#adata = sc.read_h5ad("data/SKM_mouse_raw_cells2nuclei_2022-03-30.h5ad")
#adata = sc.read_h5ad("data/SKM_mouse_pp_cells2nuclei_2022-03-30.h5ad")


In [3]:
target_gene = "FEZ2"
target_cell_type= "Adipocyte"

In [4]:
set((adata.obs['SampleID'].str.split("_").str[0]).tolist())

{'5386STDY7600836',
 '5386STDY7600837',
 '5386STDY7600838',
 '5386STDY7600839',
 '5386STDY7645353',
 '5386STDY7645354',
 '5386STDY7645355',
 '5386STDY7796286',
 '5386STDY7796287',
 '5386STDY7835292',
 '5386STDY7835293',
 '5386STDY8047212',
 '5386STDY8090404',
 '5386STDY8090405',
 '5386STDY8090406',
 '5386STDY8090407',
 '5386STDY8493510',
 '5386STDY8552613',
 '5386STDY8552614',
 '5386STDY8552709',
 'WS',
 'mus'}

In [5]:
adata.obs

,SampleID,DonorID,Age_group,Age_bin,Sex,batch,10X_version,annotation_level0,annotation_level1,annotation_level2,n_counts,n_genes,percent_mito,percent_ribo,scrublet_score
mus_SNuc7468112-GTGTGCGCAATGGACG,mus_SNuc7468112,339C,70-75,old,F,cells,3'v2,VenEC,VenEC,Vein,27233.0,4442.0,0.023281,0.023281,0.051800
mus_SNuc7468112-CACAGGCGTTGCCTCT,mus_SNuc7468112,339C,70-75,old,F,cells,3'v2,ArtEC,ArtEC,Artery,23438.0,4906.0,0.014848,0.014848,0.159170
mus_SNuc7468112-TCAGCAAAGCTGCGAA,mus_SNuc7468112,339C,70-75,old,F,cells,3'v2,VenEC,VenEC,Vein-CCL2+,22903.0,4106.0,0.021395,0.021395,0.058824
mus_SNuc7468112-GCATACACAGCTTCGG,mus_SNuc7468112,339C,70-75,old,F,cells,3'v2,VenEC,VenEC,Vein-CCL2+,22678.0,4402.0,0.020416,0.020416,0.044118
mus_SNuc7468112-GATTCAGAGTGTACGG,mus_SNuc7468112,339C,70-75,old,F,cells,3'v2,VenEC,VenEC,Vein-CCL2+,22230.0,4242.0,0.020153,0.020153,0.051800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WS_A_SKM10691779-TTTCAGTGTATAGGAT,WS_A_SKM10691779,582C,55-60,old,F,nuclei,3'v3,FB,FB,Inter_FB,508.0,422.0,0.000000,0.000000,0.061889
WS_A_SKM10691779-GATCGTATCTCCAATT,WS_A_SKM10691779,582C,55-60,old,F,nuclei,3'v3,FB,FB,Inter_FB,517.0,422.0,0.000000,0.000000,0.076923
WS_A_SKM10691779-CCGATGGAGAACGTGC,WS_A_SKM10691779,582C,55-60,old,F,nuclei,3'v3,FB,FB,Par_FB,504.0,400.0,0.000000,0.000000,0.068966
WS_A_SKM10691779-GTTGTAGGTCAAAGAT,WS_A_SKM10691779,582C,55-60,old,F,nuclei,3'v3,FB,FB,Adv_FB,501.0,405.0,0.000000,0.000000,0.055556


In [6]:
adata.var_names

Index(['MIR1302-2HG', 'AL627309.1', 'AL627309.3', 'AC114498.1', 'AL669831.2',
       'AL669831.5', 'FAM87B', 'LINC00115', 'FAM41C', 'AL645608.7',
       ...
       'AC011043.2', 'AL592183.1', 'AC007325.1', 'AC007325.4', 'AC007325.2',
       'AL354822.1', 'AC004556.1', 'AC233755.2', 'AC233755.1', 'AC240274.1'],
      dtype='object', length=29400)

In [7]:
cell_type_mask = adata.obs["annotation_level0"] == target_cell_type
#cell_type_mask = adata.obs["annotation"] == target_cell_type


In [8]:
if target_gene not in adata.var_names:
    raise ValueError(f"Gene {target_gene} not found in the dataset.")


In [9]:
expression_data = adata[cell_type_mask, adata.var_names == adata.var_names].X


In [10]:
#len(adata.X.toarray())

In [11]:
#len(expression_data.toarray())

In [12]:
expression_data

<137x29400 sparse matrix of type '<class 'numpy.float32'>'
	with 251974 stored elements in Compressed Sparse Row format>

In [13]:
mean_expression = expression_data.mean()
mean_expression

0.09497519

In [14]:
unique_cell_types = adata.obs["annotation_level0"].unique()
#unique_cell_types = adata.obs["annotation"].unique()

unique_cell_types

['VenEC', 'ArtEC', 'LymphEC', 'CapEC', 'SMC', ..., 'MF-I', 'MF-II', 'Adipocyte', 'MF-Isn(fg)', 'MF-IIsn(fg)']
Length: 36
Categories (36, object): ['Adipocyte', 'ArtEC', 'B-cell', 'B-plasma', ..., 'cDC2', 'mSchwann', 'nmSchwann', 'pDC']

In [15]:
cell_types = unique_cell_types.tolist()

In [16]:
len(cell_types)

36

In [17]:
cell_type_counts = adata.obs["annotation_level0"].value_counts()
#cell_type_counts = adata.obs["annotation"].value_counts()

total_cells = len(adata.obs)
cell_type_proportions = {cell_type: count / total_cells for cell_type, count in cell_type_counts.items()}
cell_type_proportions

{'MF-I': 0.22959036039331518,
 'MF-II': 0.15591747151413238,
 'FB': 0.15246149562406844,
 'MuSC': 0.11036738170243665,
 'SMC': 0.04394494461157124,
 'T-cell': 0.04031971871741255,
 'MF-IIsc(fg)': 0.038627218676464965,
 'MF-Isc(fg)': 0.02841762165526504,
 'Macrophage': 0.02578605707546912,
 'NK-cell': 0.020337298879128198,
 'CapEC': 0.01834997625040265,
 'Specialised MF': 0.017935040756492923,
 'Monocyte': 0.016324435878816997,
 'VenEC': 0.015500024568548982,
 'Pericyte': 0.015270718111388342,
 'ArtEC': 0.013862121303115838,
 'B-cell': 0.007239532433214495,
 'cDC2': 0.006322306604571934,
 'Neutrophil': 0.0053231856126577165,
 'PnFB': 0.00509933883304852,
 'Tenocyte': 0.004717161404447453,
 'EnFB': 0.004597048498315689,
 'LymphEC': 0.004198492037060291,
 'MF-Isn(fg)': 0.003685282347224573,
 'Mast': 0.0034450565349610452,
 'mSchwann': 0.0026534032900016923,
 'Hyb': 0.0022603065062977382,
 'B-plasma': 0.001801693591976458,
 'MF-IIsn(fg)': 0.0014085968082725034,
 'Mesothelium': 0.0011083145

In [20]:
valid_genes = list(adata.var_names)


In [37]:
genes = ["NT5C2"
"ALDOA",
"BLCAP",
"FEZ2",
"SLC16A3",
"STUM",
"STUM",
"CA3",
"STIM1",
"ACIN1",
"HNRNPM",
"TPM3",
"CALM1",
"EHMT1",
"MYL2",
"RPS24",
"ENO3",
"TNNI1",
"EEF2",
"HBA2",
"MB",
"RPL13A",
"KL",
"GPX7",
"NDUFB4",
"KL",
"TAF9B",
"AK1",
"GLB1L",
"ADA",
"AK1",
"GLB1L",
"ADA",
"ERI3",
"RNF7",
"RECQL",
"CLIC4",
"AS3MT",
"CLIC6",
"MAST1",
"CLIC4",
"CCNI",
"MMP23B",
"AK1",
"CLIC4",
"CUTC",
"KLRB1",
"MICU1",
"PARK7",
"SBDS",
"MEIS2",
"ALDOA",
"CA3",
"FEZ2",
"EIF3C",
"UBE2H",
"CALM1"]  # Replace with your list of genes
genes = genes = ["NT5C2"]
# Ensure all genes are in the dataset
valid_genes = [gene for gene in genes if gene in adata.var_names]
missing_genes = set(genes) - set(valid_genes)
if missing_genes:
    print(f"Warning: The following genes are not in the dataset and will be skipped: {missing_genes}")



In [38]:
sc.get.obs_df(adata, keys=valid_genes)

,NT5C2
mus_SNuc7468112-GTGTGCGCAATGGACG,0.000000
mus_SNuc7468112-CACAGGCGTTGCCTCT,0.000000
mus_SNuc7468112-TCAGCAAAGCTGCGAA,0.000000
mus_SNuc7468112-GCATACACAGCTTCGG,0.365307
mus_SNuc7468112-GATTCAGAGTGTACGG,0.000000
...,...
WS_A_SKM10691779-TTTCAGTGTATAGGAT,0.000000
WS_A_SKM10691779-GATCGTATCTCCAATT,0.000000
WS_A_SKM10691779-CCGATGGAGAACGTGC,0.000000
WS_A_SKM10691779-GTTGTAGGTCAAAGAT,0.000000


In [39]:
sc.get.obs_df(adata, keys=['annotation_level0'])

,annotation_level0
mus_SNuc7468112-GTGTGCGCAATGGACG,VenEC
mus_SNuc7468112-CACAGGCGTTGCCTCT,ArtEC
mus_SNuc7468112-TCAGCAAAGCTGCGAA,VenEC
mus_SNuc7468112-GCATACACAGCTTCGG,VenEC
mus_SNuc7468112-GATTCAGAGTGTACGG,VenEC
...,...
WS_A_SKM10691779-TTTCAGTGTATAGGAT,FB
WS_A_SKM10691779-GATCGTATCTCCAATT,FB
WS_A_SKM10691779-CCGATGGAGAACGTGC,FB
WS_A_SKM10691779-GTTGTAGGTCAAAGAT,FB


In [ ]:
# Initialize the results dictionary
expression_data = []

# Loop through each cell type and gene
for cell_type in cell_types:
    cell_mask = adata.obs["annotation_level0"] == cell_type
    
    for gene in valid_genes:
        mean_expression = adata[cell_mask, adata.var_names == gene].X.mean()
        expression_data.append({"Cell Type": cell_type, "Gene": gene, "Mean Expression": mean_expression})

# Convert to a Pandas DataFrame
expression_df = pd.DataFrame(expression_data)

In [ ]:
expression_df

In [ ]:
expression_df=expression_df.drop_duplicates()

In [ ]:
expression_df.to_csv("mean_expression_gene_cell_type.csv")

In [ ]:
reshaped_df = expression_df.pivot(index="Gene", columns="Cell Type", values="Mean Expression")

# Optionally, sort the index and columns for better organization
reshaped_df = reshaped_df.sort_index().sort_index(axis=1)

In [ ]:
reshaped_df["most abundant in"] = reshaped_df.idxmax(axis=1)


In [ ]:
reshaped_df["most abundant in"]

In [ ]:
reshaped_df.to_csv("results/mean_expression_cell.csv")

In [ ]:
proportion_df = pd.DataFrame.from_dict(cell_type_proportions, orient="index", columns=["Proportion"])

# Reset the index to make the cell type a column (optional)
proportion_df = proportion_df.reset_index().rename(columns={"index": "Cell Type"})
proportion_df = proportion_df.set_index("Cell Type")

In [ ]:
proportion_df.to_csv("results/proportion_cell_type.csv")